In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
dataset_path = "//kaggle/input/datasets/navneet0094/jafee-dataset/"

print(os.listdir(dataset_path))

In [ ]:
from PIL import Image

data = []
labels = []

emotion_map = {
    'AN':0, 'DI':1, 'FE':2,
    'HA':3, 'SA':4, 'SU':5, 'NE':6
}

dataset_path = "/kaggle/input/datasets/navneet0094/jafee-dataset/jaffe"

for img_name in os.listdir(dataset_path):
    img_path = os.path.join(dataset_path, img_name)

    try:
        img = Image.open(img_path).convert('L')  # grayscale
        img = img.resize((48,48))
        img = np.array(img)

        emotion = img_name.split('.')[1][:2]
        label = emotion_map[emotion]

        data.append(img)
        labels.append(label)

    except Exception as e:
        print("Error loading:", img_name)

data = np.array(data)
labels = np.array(labels)

print("Data shape:", data.shape)
print("Labels:", np.unique(labels))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))

for i in range(5):
    plt.subplot(1,5,i+1)
    plt.imshow(data[i], cmap='gray')
    plt.title(f"Label: {labels[i]}")
    plt.axis('off')

plt.show()

In [ ]:
# HOG
from skimage.feature import hog

X_hog = []

for img in data:
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2)
    )
    X_hog.append(features)

X_hog = np.array(X_hog)

print("HOG feature shape:", X_hog.shape)

In [ ]:
# HOG vis
sample_img = data[0]

features, hog_image = hog(
    sample_img,
    orientations=9,
    pixels_per_cell=(8,8),
    cells_per_block=(2,2),
    visualize=True
)

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.imshow(sample_img, cmap='gray')
plt.title("Original")

plt.subplot(1,2,2)
plt.imshow(hog_image, cmap='gray')
plt.title("HOG Image")

plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import time

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_hog, labels, test_size=0.3, random_state=42,shuffle=True
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  
X_test = scaler.transform(X_test)         

start = time.time()

model = SVC(kernel='linear')
model.fit(X_train, y_train)

end = time.time()

y_pred = model.predict(X_test)

hog_acc = accuracy_score(y_test, y_pred)
hog_time = end - start

print("HOG Accuracy:", hog_acc)
print("HOG Time:", hog_time)

In [ ]:
emotion_names = ['anger','disgust','fear','happy','sadness','surprise','neutral']
print(classification_report(y_test, y_pred, target_names=emotion_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("HOG Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear'))
])

cv_scores = cross_val_score(pipeline, X_hog, labels, cv=10)

print("\n=== 10-Fold CV ===")
print("Fold Accuracies:", cv_scores)
print("Mean Accuracy:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

In [ ]:
plt.plot(cv_scores, marker='o')
plt.title("10-Fold CV Accuracy (HOG)")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

# get predictions from CV
y_pred_cv = cross_val_predict(pipeline, X_hog, labels, cv=10)

# confusion matrix
cm = confusion_matrix(labels, y_pred_cv)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("CV Confusion Matrix (HOG)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:

# classification report
print(classification_report(labels, y_pred_cv, target_names=emotion_names))

In [ ]:
import cv2
# SIFT
sift = cv2.SIFT_create()

X_sift = []

for img in data:
    img = img.astype('uint8')

    _, des = sift.detectAndCompute(img, None)

    if des is None:
        des = np.zeros(500)
    else:
        des = des.flatten()
        if len(des) > 500:
            des = des[:500]
        else:
            des = np.pad(des, (0, 500-len(des)))

    X_sift.append(des)

X_sift = np.array(X_sift)

print("SIFT feature shape:", X_sift.shape)

In [ ]:

X_scaled = StandardScaler().fit_transform(X_sift)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, labels, test_size=0.2, random_state=42,shuffle=False
)

start = time.time()

model = SVC(kernel='linear')
model.fit(X_train, y_train)

end = time.time()

y_pred1 = model.predict(X_test)

sift_acc = accuracy_score(y_test, y_pred1)
sift_time = end - start

print("SIFT Accuracy:", sift_acc)
print("SIFT Time:", sift_time)

In [ ]:
emotion_names = ['anger','disgust','fear','happy','sadness','surprise','neutral']
print(classification_report(y_test, y_pred1, target_names=emotion_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred1)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("Sift Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear'))
])

cv_scores = cross_val_score(pipeline,X_sift , labels, cv=10)

print("\n=== 10-Fold CV ===")
print("Fold Accuracies:", cv_scores)
print("Mean Accuracy:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

In [ ]:
# LBP
from skimage.feature import local_binary_pattern

X_lbp = []

for img in data:
    lbp = local_binary_pattern(img, 8, 1, method="uniform")
    # hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    hist, _ = np.histogram(lbp.ravel(), bins=256, range=(0,256))
    X_lbp.append(hist)

X_lbp = np.array(X_lbp)

print("LBP feature shape:", X_lbp.shape)

In [ ]:
X_scaled = StandardScaler().fit_transform(X_lbp)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, labels, test_size=0.2, random_state=42 ,shuffle =True
)

start = time.time()

model = SVC(kernel='linear')
model.fit(X_train, y_train)

end = time.time()

y_pred2 = model.predict(X_test)

lbp_acc = accuracy_score(y_test, y_pred2)
lbp_time = end - start

print("LBP Accuracy:", lbp_acc)
print("LBP Time:", lbp_time)

In [ ]:
emotion_names = ['anger','disgust','fear','happy','sadness','surprise','neutral']
print(classification_report(y_test, y_pred2, target_names=emotion_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred2)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("LBP Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear'))
])

cv_scores = cross_val_score(pipeline,X_lbp , labels, cv=10)

print("\n=== 10-Fold CV ===")
print("Fold Accuracies:", cv_scores)
print("Mean Accuracy:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

In [ ]:
# lightbgm
# xgboost 
# random forest 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

start = time.time()

rf_model.fit(X_train, y_train)

end = time.time()

y_pred_rf = rf_model.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("RF Time:", end - start)




In [ ]:
# report
print(classification_report(y_test, y_pred_rf, target_names=emotion_names))


In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_scores = cross_val_score(rf_model, X_hog, labels, cv=10)

print("\n=== Random Forest CV ===")
print("Scores:", rf_scores)
print("Mean:", rf_scores.mean())
print("Std:", rf_scores.std())

In [ ]:
from sklearn.model_selection import cross_val_predict

y_pred_rf_cv = cross_val_predict(rf_model, X_hog, labels, cv=10)

cm_rf = confusion_matrix(labels, y_pred_rf_cv)

plt.figure(figsize=(6,5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("Random Forest CV Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(labels, y_pred_rf_cv, target_names=emotion_names))

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    # use_label_encoder=False,
    eval_metric='mlogloss'
)

start = time.time()

xgb_model.fit(X_train, y_train)

end = time.time()

y_pred_xgb = xgb_model.predict(X_test)

print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGB Time:", end - start)


In [ ]:

print(classification_report(y_test, y_pred_xgb, target_names=emotion_names))



In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    # use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_scores = cross_val_score(xgb_model, X_hog, labels, cv=10)

print("\n=== XGBoost CV ===")
print("Scores:", xgb_scores)
print("Mean:", xgb_scores.mean())
print("Std:", xgb_scores.std())

In [ ]:
y_pred_xgb_cv = cross_val_predict(xgb_model, X_hog, labels, cv=10)

cm_xgb = confusion_matrix(labels, y_pred_xgb_cv)

plt.figure(figsize=(6,5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("XGBoost CV Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(labels, y_pred_xgb_cv, target_names=emotion_names))

In [ ]:
!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,
    verbose=-1  
)

start = time.time()

lgb_model.fit(X_train, y_train)

end = time.time()

y_pred_lgb = lgb_model.predict(X_test)

print("LGB Accuracy:", accuracy_score(y_test, y_pred_lgb))
print("LGB Time:", end - start)


In [ ]:

print(classification_report(y_test, y_pred_lgb, target_names=emotion_names))



In [ ]:
cm = confusion_matrix(y_test, y_pred_lgb)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("LightGBM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,
    verbose=-1   # suppress logs
)

lgb_scores = cross_val_score(lgb_model, X_hog, labels, cv=10)

print("\n=== LightGBM CV ===")
print("Scores:", lgb_scores)
print("Mean:", lgb_scores.mean())
print("Std:", lgb_scores.std())

In [ ]:
y_pred_lgb_cv = cross_val_predict(lgb_model, X_hog, labels, cv=10)

cm_lgb = confusion_matrix(labels, y_pred_lgb_cv)

plt.figure(figsize=(6,5))
sns.heatmap(cm_lgb, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names,
            yticklabels=emotion_names)

plt.title("LightGBM CV Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(labels, y_pred_lgb_cv, target_names=emotion_names))

In [ ]:
methods = ['HOG+SVM', 'RandomForest', 'XGBoost', 'LightGBM']
accuracy = [
    hog_acc,
    accuracy_score(y_test, y_pred_rf),
    accuracy_score(y_test, y_pred_xgb),
    accuracy_score(y_test, y_pred_lgb)
]

plt.bar(methods, accuracy)
plt.title("Model Comparison (JAFFE)")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

In [ ]:
methods = ['SVM', 'RF', 'XGB', 'LGB']
means = [
    cv_scores.mean(),
    rf_scores.mean(),
    xgb_scores.mean(),
    lgb_scores.mean()
]

plt.bar(methods, means)
plt.title("Model Comparison (10-Fold CV)")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

In [ ]:
plt.boxplot(
    [cv_scores, rf_scores, xgb_scores, lgb_scores],
    labels=['SVM','RF','XGB','LGB']
)

plt.title("Accuracy Distribution (CV)")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
plt.plot(cv_scores, marker='o', label='SVM')
plt.plot(rf_scores, marker='o', label='RF')
plt.plot(xgb_scores, marker='o', label='XGB')
plt.plot(lgb_scores, marker='o', label='LGB')

plt.legend()
plt.title("Fold-wise Accuracy Comparison")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.grid()
plt.show()